# Setting up and cleaning the data:

In [2]:
#STEP 1
import pandas as pd
import geopandas as gpd

FILES = [
    "../data/remplissage_niddepoule_2024.gpkg", 
    "../data/remplissage_niddepoule_2025.gpkg",
]
YEARS = [2024, 2025]
print("GeoPandas", gpd.__version__, "is ready")

GeoPandas 1.2.0 is ready


In [3]:
#STEP 2
frames = []
for path in FILES:   # reading the two .gpkg files for the years 2024 & 2025
    g = gpd.read_file(path)
    print(path)
    print("  rows:", len(g), "| columns:", g.columns.tolist(), "| system:", g.crs)
    print("  dates:", g["Date"].min(), "to", g["Date"].max())
    frames.append(g)

../data/remplissage_niddepoule_2024.gpkg
  rows: 50411 | columns: ['Véhicule', 'Date', 'geometry'] | system: EPSG:2950
  dates: 2023-04-11 09:04:57.063000 to 2024-05-30 23:56:08.063000
../data/remplissage_niddepoule_2025.gpkg
  rows: 74159 | columns: ['Véhicule', 'Date', 'geometry'] | system: EPSG:2950
  dates: 2025-01-18 21:29:15.063 to 2025-05-20 15:45:54.063


In [4]:
# STEP 3: cleaning data (stacking the years, plain names, count what I drop)
both = pd.concat(frames, ignore_index=True)   # stacking 2024 and 2025 into one table
n_read = len(both)                            # remembering how many rows we started with

# this builds a plain date and year from the Date column (dataset originally holds date and time together)
when = pd.to_datetime(both["Date"], format="ISO8601")   # turn the text into real dates
both["date"] = when.dt.strftime("%Y-%m-%d")             # date only, like 2025-04-04
both["year"] = when.dt.year                             # the year, as a number
both = both[["date", "year", "geometry"]]               # keeping only what I want for my map

# droping rows with no location or no date, and it counts them
bad = both.geometry.isna() | both.geometry.is_empty | when.isna()
n_bad = int(bad.sum())
both = both[~bad].copy()
both["year"] = both["year"].astype(int)

# dropping rows from any year other than 2024 and 2025, and count them
other = ~both["year"].isin(YEARS)
n_other = int(other.sum())
both = both[~other].copy()

print("rows read:", n_read)
print("dropped, no location or no date:", n_bad)
print("dropped, year not in", YEARS, ":", n_other)
print("rows kept:", len(both))

rows read: 124570
dropped, no location or no date: 0
dropped, year not in [2024, 2025] : 4
rows kept: 124566


# Converting meters to degree

In [5]:
# STEP 4: metres to degrees (the files use metres, web maps need degrees)
both = both.to_crs("EPSG:4326")   # convert from EPSG:2950 to latitude and longitude
print(both.crs)                   # should say EPSG:4326; points should now look like POINT (-73.6 45.5)
both.head()                       

EPSG:4326


,date,year,geometry
0,2024-05-13,2024,POINT (-73.61362 45.52617)
1,2024-05-13,2024,POINT (-73.61354 45.52622)
2,2024-05-13,2024,POINT (-73.61331 45.52637)
3,2024-05-13,2024,POINT (-73.61349 45.52632)
4,2024-05-13,2024,POINT (-73.61395 45.52548)


In [ ]:
# STEP 5: here I find the busiest areas to help narrow down where to cut the map
lat = both.geometry.y.round(2)   # latitude of each stop, rounded to about 1 km squares
lon = both.geometry.x.round(2)   # longitude of each stop rounded the same way
# counting the stops in each square and showing the ten busiest
print(both.groupby([lat, lon]).size().sort_values(ascending=False).head(10))

45.51  -73.56    3116
       -73.57    2138
45.53  -73.61    1922
45.58  -73.59    1705
45.57  -73.55    1704
45.55  -73.58    1666
       -73.53    1421
45.50  -73.57    1417
45.52  -73.60    1415
       -73.61    1409
dtype: int64


In [7]:
# Optional check for step 5:
# same ten busiest squares, with every latitude shown
busy = both.groupby([lat, lon]).size().sort_values(ascending=False).head(10)   # ten busiest 1 km squares
busy = busy.rename_axis(["lat", "lon"]).reset_index(name="stops")              # turns it into a plain table
print(busy.to_string(index=False))

  lat    lon  stops
45.51 -73.56   3116
45.51 -73.57   2138
45.53 -73.61   1922
45.58 -73.59   1705
45.57 -73.55   1704
45.55 -73.58   1666
45.55 -73.53   1421
45.50 -73.57   1417
45.52 -73.60   1415
45.52 -73.61   1409


In [8]:
# STEP 5b: counting the stops in each month, per year, to see which months each year really covers
month = pd.to_datetime(both["date"], format="ISO8601").dt.to_period("M")   # turns each date into its month, like 2025-03
by_month = both.groupby(["year", month]).size()                            # counts the rows for each year and month pair
print(by_month.rename("stops").reset_index().to_string(index=False))       # print every row so nothing is hidden

 year    date  stops
 2024 2024-01  15783
 2024 2024-03  28518
 2024 2024-04   2350
 2024 2024-05   3756
 2025 2025-01  13942
 2025 2025-02   1560
 2025 2025-03  29973
 2025 2025-04  28399
 2025 2025-05    285


In [ ]:
# FIRST AND LAST DATES: checking earliest and latest stop in each year, across the whole city (before the box cut)
for y in YEARS:                                   # YEARS is [2024, 2025] from Step 1
    d = both[both["year"] == y]["date"]           # the dates for that year only
    print(y, "first:", d.min(), "| last:", d.max())

2024 first: 2024-01-17 | last: 2024-05-30
2025 first: 2025-01-18 | last: 2025-05-20


# Cutting to One Area - a box in degrees:

In [10]:
SOUTH, NORTH = 45.495, 45.535   # bottom and top of the box (latitude); built from ny busiest squares table
WEST, EAST = -73.615, -73.555   # left and right sides of the box (longitude)

print("before:", len(both))
area = both.cx[WEST:EAST, SOUTH:NORTH].copy()   # keeping only the stops inside the box
n_outside = len(both) - len(area)               # how many has been cut out 
print("after:", len(area), "| outside the box:", n_outside)
for y, n in area["year"].value_counts().sort_index().items():   # stops per year
    print("  ", y, ":", n)

before: 124566
after: 21205 | outside the box: 103361
   2024 : 8764
   2025 : 12441


# Writing the GeoJSON file for the boxed area 

COORDINATE_PRECISION = 6 is used.

In [11]:
# STEP 7: saving the cut-down stops as a GeoJSON file for the web map
import os                                              

OUT = "../data/potholes_2024_2025.geojson"             # where the file goes (data folder)

area.to_file(OUT, driver="GeoJSON", COORDINATE_PRECISION=6)   # writing the file, keeping 6 decimals per coordinate

size_mb = os.path.getsize(OUT) / 1_000_000             # file size in megabytes
print("saved:", OUT)
print("rows written:", len(area), "| file size:", round(size_mb, 2), "MB")

saved: ../data/potholes_2024_2025.geojson
rows written: 21205 | file size: 3.22 MB


# The check cell for Central Montreal Potholes Data (final check)

In [ ]:
FILE = "../data/potholes_2024_2025.geojson"      # the file we saved in Step 7
WEST, EAST, SOUTH, NORTH = -73.615, -73.555, 45.495, 45.535   # the box from Step 6, in degrees
EXPECTED = 21205                                 # rows we expect (the "after" count from Step 6)
POPUP_FIELDS = ["date", "year"]                  # columns the map will read for its popup
YEARS = [2024, 2025]                             # years the year switch needs
# ---------------------------------------------------------

import os                                        
import geopandas as gpd                          

back = gpd.read_file(FILE)
problems = []                                    # every failed check gets added to this list

# check 1: the row count matches what we expect
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")

# check 2: the coordinates are in longitude and latitude (degrees)
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")

# check 3: every point sits inside the box
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")

# check 4: every stop is a single point
types = sorted(back.geom_type.unique())
if types != ["Point"]:
    problems.append(f"Expected only Point, found {types}.")

# check 5: the popup columns exist
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")

# check 6: both years are in the file (as the year switch needs them)
years_found = sorted(back["year"].unique().tolist()) if "year" in back.columns else []
if years_found != YEARS:
    problems.append(f"Years in the file are {years_found}, but the map needs {YEARS}.")

# check 7: the file is small enough for the page to load fast
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")

# print the result
print("Geometry types:", types)
print("Years found:", years_found)
print("File size:", round(mb, 2), "MB")
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Point']
Years found: [2024, 2025]
File size: 3.22 MB
All checks passed


In [ ]:
# Point Accuracy CHECK: how far did rounding and saving move the points on the Map? (saved file vs the table in memory)
saved = gpd.read_file(OUT)                                    
dx = (saved.geometry.x.values - area.geometry.x.values) * 78000    # east-west difference in metres at this latitude
dy = (saved.geometry.y.values - area.geometry.y.values) * 111000   # north-south difference in metres
print("largest shift in metres:", round(max(abs(dx).max(), abs(dy).max()), 3))

largest shift in metres: 0.055


This further confirms my code did not cause the offset. GPS error most likely caused some points to be drawn inside buildings.